# explore_external_location_and_tables

**Source:** `06_analysis/explore_external_location_and_tables.py`  
**Purpose:** Databricks notebook auto-generated from framework Python module.


## Section 1: Additional module code and configuration

This cell handles: *Additional module code and configuration*


In [ ]:
"""
Explore external location files and view transformed table content.

Use this notebook to:
1. List files in the external location
2. Query UC tables to see transformation results
3. Inspect schema and row counts after pipeline execution
"""

from __future__ import annotations

import re
from typing import Any


## Section 2: Define `_get_active_spark()` function with logic for processing

This cell handles: *Define `_get_active_spark()` function with logic for processing*


In [ ]:
def _get_active_spark() -> Any:
    """Return active Spark session or raise a clear runtime error."""
    existing = globals().get("spark")
    if existing is not None:
        return existing

    try:
        from pyspark.sql import SparkSession
    except Exception as exc:  # pragma: no cover - runtime environment guard
        raise RuntimeError(
            "PySpark is not available. Run this notebook in Databricks."
        ) from exc

    session = SparkSession.getActiveSession()
    if session is None:
        raise RuntimeError("No active Spark session found. Attach this notebook to a cluster.")
    return session


## Section 3: Define `_get_dbutils()` function with logic for processing

This cell handles: *Define `_get_dbutils()` function with logic for processing*


In [ ]:
def _get_dbutils() -> Any:
    """Return dbutils or raise a clear runtime error."""
    value = globals().get("dbutils")
    if value is None:
        raise RuntimeError("dbutils is not available. Run this notebook in Databricks.")
    return value


spark = _get_active_spark()
dbutils = _get_dbutils()

# COMMAND ----------

# MAGIC %md
# MAGIC # 1. List Files in External Location

# COMMAND ----------

# List all files recursively in the external location
external_location_path = "abfss://rngpub@adlsdnapdevsilver.dfs.core.windows.net/eng511"


## Section 4: Define `list_files_recursive()` function with logic for processing

This cell handles: *Define `list_files_recursive()` function with logic for processing*


In [ ]:
def list_files_recursive(path, max_depth=3, current_depth=0):
    """Recursively list files and folders in external location."""
    if current_depth >= max_depth:
        return
    
    try:
        items = dbutils.fs.ls(path)
        for item in items:
            indent = "  " * current_depth
            if item.isDir():
                print(f"{indent}📁 {item.name} (folder)")
                list_files_recursive(item.path, max_depth, current_depth + 1)
            else:
                size_mb = item.size / (1024 * 1024)
                print(f"{indent}📄 {item.name} ({size_mb:.2f} MB)")
    except Exception as e:
        print(f"Error listing {path}: {e}")

print(f"Files in external location: {external_location_path}\n")
list_files_recursive(external_location_path)

# COMMAND ----------

# MAGIC %md
# MAGIC # 1b. Validate Folder Pattern and snappy.parquet Files

# COMMAND ----------


## Section 5: Define `_short_error()` function with logic for processing

This cell handles: *Define `_short_error()` function with logic for processing*


In [ ]:
def _short_error(exc: Exception) -> str:
    """Return concise error text without the JVM stacktrace tail."""
    msg = str(exc)
    marker = "JVM stacktrace:"
    if marker in msg:
        return msg.split(marker, 1)[0].strip()
    return msg


## Section 6: Define `_safe_list_dirs()` function with logic for processing

This cell handles: *Define `_safe_list_dirs()` function with logic for processing*


In [ ]:
def _safe_list_dirs(path: str):
    """List only child directories. Returns [] when path is not found."""
    try:
        return [x for x in dbutils.fs.ls(path) if x.isDir()]
    except Exception as exc:
        print(f"Could not list {path}: {_short_error(exc)}")
        return []


## Section 7: Define `_resolve_scan_roots()` function with logic for processing

This cell handles: *Define `_resolve_scan_roots()` function with logic for processing*


In [ ]:
def _resolve_scan_roots(base_path: str, preferred_parent: str) -> list[str]:
    """Resolve where to scan.

    If base_path/preferred_parent exists, scan it.
    Otherwise, scan all top-level directories under base_path.
    """
    preferred_root = f"{base_path.rstrip('/')}/{preferred_parent}"
    try:
        dbutils.fs.ls(preferred_root)
        return [preferred_root]
    except Exception:
        print(
            f"WARN: preferred folder '{preferred_parent}' not found under {base_path}. "
            "Falling back to all top-level folders."
        )

    top_level_dirs = _safe_list_dirs(base_path)
    return [d.path.rstrip("/") for d in top_level_dirs]


## Section 8: Define `summarize_parquet_layout()` function with logic for processing

This cell handles: *Define `summarize_parquet_layout()` function with logic for processing*


In [ ]:
def summarize_parquet_layout(base_path, expected_parent="connect", expected_child_folders=15):
    """Summarize folder layout and parquet file pattern for quick parity checks."""
    scan_roots = _resolve_scan_roots(base_path, expected_parent)
    if not scan_roots:
        print(f"No folders found under base path: {base_path}")
        return

    for root in scan_roots:
        root_name = root.rstrip("/").split("/")[-1]
        print(f"\nChecking parent folder: {root}")

        children = _safe_list_dirs(root)
        print(f"Found {len(children)} child folders under '{root_name}'")

        if root_name == expected_parent:
            if len(children) == expected_child_folders:
                print(f"PASS: child folder count matches expected value {expected_child_folders}")
            else:
                print(f"WARN: child folder count differs from expected {expected_child_folders}")

        all_have_snappy = True
        total_snappy_files = 0

        for child in children:
            try:
                files = dbutils.fs.ls(child.path)
                names = [f.name for f in files if not f.isDir()]
                snappy_files = [n for n in names if n.endswith(".snappy.parquet")]
                delta_logs = [n for n in names if n.startswith("_delta_log")]
                total_snappy_files += len(snappy_files)

                if not snappy_files:
                    all_have_snappy = False

                print(
                    f"- {child.name.rstrip('/')}: "
                    f"snappy_parquet={len(snappy_files)}, delta_log_entries={len(delta_logs)}"
                )
            except Exception as exc:
                all_have_snappy = False
                print(f"- {child.name.rstrip('/')}: ERROR while listing files: {_short_error(exc)}")

        print("Summary:")
        print(f"Total .snappy.parquet files found: {total_snappy_files}")
        print(f"All child folders contain snappy parquet: {all_have_snappy}")
        print("Note: Presence of _delta_log indicates Delta tables (which store data in parquet files).")


summarize_parquet_layout(external_location_path, expected_parent="connect", expected_child_folders=15)

# COMMAND ----------

# MAGIC %md
# MAGIC # 1c. Strict Parquet Pattern Validation

# COMMAND ----------

PARQUET_FILE_PATTERN = re.compile(r".*\.snappy\.parquet$")


## Section 9: Define `validate_parquet_pattern()` function with logic for processing

This cell handles: *Define `validate_parquet_pattern()` function with logic for processing*


In [ ]:
def validate_parquet_pattern(base_path, parent_folder="connect"):
    """Validate parquet naming pattern across immediate child folders."""
    scan_roots = _resolve_scan_roots(base_path, parent_folder)
    if not scan_roots:
        print(f"ERROR: no folders found under {base_path}")
        return

    for root in scan_roots:
        print(f"\nValidating parquet files under: {root}")
        children = _safe_list_dirs(root)

        total_files = 0
        pattern_matches = 0
        folders_without_match: list[str] = []

        for child in children:
            files = [f for f in dbutils.fs.ls(child.path) if not f.isDir()]
            parquet_files = [f.name for f in files if f.name.endswith(".parquet")]
            matched = [name for name in parquet_files if PARQUET_FILE_PATTERN.match(name)]

            total_files += len(parquet_files)
            pattern_matches += len(matched)

            if parquet_files and not matched:
                folders_without_match.append(child.name.rstrip("/"))

        print(f"Total parquet files: {total_files}")
        print(f"Files matching '*.snappy.parquet': {pattern_matches}")

        if folders_without_match:
            print("WARN: These folders had parquet files but none matched '*.snappy.parquet':")
            for name in folders_without_match:
                print(f"- {name}")
        else:
            print("PASS: All folders with parquet files match the '*.snappy.parquet' pattern.")


validate_parquet_pattern(external_location_path, parent_folder="connect")

# COMMAND ----------

# MAGIC %md
# MAGIC # 2. Configure UC Catalog and Schemas

# COMMAND ----------

# Set up your UC context (update with actual values)
catalog = spark.conf.get("spark.databricks.udp.uc_catalog", "main")
bronze_schema = spark.conf.get("spark.databricks.udp.uc_bronze_schema", "bronze_dev")
silver_schema = spark.conf.get("spark.databricks.udp.uc_silver_schema", "silver_dev")
conformance_schema = "bronze_dev"  # Usually bronze layer

print(f"Catalog: {catalog}")
print(f"Bronze Schema: {bronze_schema}")
print(f"Silver Schema: {silver_schema}")
print(f"Conformance Schema: {conformance_schema}")

# COMMAND ----------

# MAGIC %md
# MAGIC # 3. View Landing Table (Raw Data)

# COMMAND ----------

# Query landing table to see raw ingested data
landing_table_examples = [
    f"{catalog}.{bronze_schema}.connect_countryriskdet_landing",
    f"{catalog}.{bronze_schema}.pia_item_landing",
]

for table_fqn in landing_table_examples:
    try:
        row_count = spark.sql(f"SELECT COUNT(*) as record_count FROM {table_fqn}").collect()[0][0]
        print(f"\n{'='*80}")
        print(f"TABLE: {table_fqn}")
        print(f"Row Count: {row_count:,}")
        print(f"{'='*80}")
        
        if row_count > 0:
            spark.sql(f"SELECT * FROM {table_fqn} LIMIT 5").display()
        else:
            print("(Empty table)")
    except Exception as e:
        print(f"Table not found or error: {e}")

# COMMAND ----------

# MAGIC %md
# MAGIC # 4. View Conformance Table (After Column Mapping)

# COMMAND ----------

# Query conformance table to see transformation results
conformance_table_examples = [
    f"{catalog}.{bronze_schema}.connect_countryriskdet_conformance",
    f"{catalog}.{bronze_schema}.pia_item_conformance",
]

for table_fqn in conformance_table_examples:
    try:
        row_count = spark.sql(f"SELECT COUNT(*) as record_count FROM {table_fqn}").collect()[0][0]
        print(f"\n{'='*80}")
        print(f"TABLE: {table_fqn}")
        print(f"Row Count: {row_count:,}")
        print(f"{'='*80}")
        
        if row_count > 0:
            spark.sql(f"SELECT * FROM {table_fqn} LIMIT 5").display()
        else:
            print("(Empty table)")
    except Exception as e:
        print(f"Table not found or error: {e}")

# COMMAND ----------

# MAGIC %md
# MAGIC # 5. View Silver Table (Final Published Data)

# COMMAND ----------

# Query silver table to see final published data
silver_table_examples = [
    f"{catalog}.{silver_schema}.connect_countryriskdet",
    f"{catalog}.{silver_schema}.pia_item",
]

for table_fqn in silver_table_examples:
    try:
        row_count = spark.sql(f"SELECT COUNT(*) as record_count FROM {table_fqn}").collect()[0][0]
        print(f"\n{'='*80}")
        print(f"TABLE: {table_fqn}")
        print(f"Row Count: {row_count:,}")
        print(f"{'='*80}")
        
        if row_count > 0:
            schema_df = spark.sql(f"DESCRIBE {table_fqn}")
            print("\nSchema:")
            schema_df.display()
            
            print("\nSample data:")
            spark.sql(f"SELECT * FROM {table_fqn} LIMIT 5").display()
        else:
            print("(Empty table)")
    except Exception as e:
        print(f"Table not found or error: {e}")

# COMMAND ----------

# MAGIC %md
# MAGIC # 5b. Check Whether Silver Table Storage Is Under This External Location

# COMMAND ----------

for table_fqn in silver_table_examples:
    try:
        detail = spark.sql(f"DESCRIBE DETAIL {table_fqn}").collect()[0].asDict()
        location = str(detail.get("location", ""))
        in_external_location = location.startswith(external_location_path)

        print(f"\nTable: {table_fqn}")
        print(f"Location: {location}")
        print(f"Under external location root: {in_external_location}")
    except Exception as e:
        print(f"Could not resolve storage location for {table_fqn}: {e}")

# COMMAND ----------

# MAGIC %md
# MAGIC # 6. Quick SQL Queries for Table Inspection

# COMMAND ----------

# MAGIC %sql
# MAGIC -- Show all tables in bronze schema
# MAGIC SHOW TABLES IN main.bronze_dev;

# COMMAND ----------

# MAGIC %sql
# MAGIC -- Show all tables in silver schema
# MAGIC SHOW TABLES IN main.silver_dev;

# COMMAND ----------

# MAGIC %sql
# MAGIC -- Check table statistics
# MAGIC DESCRIBE EXTENDED main.silver_dev.connect_countryriskdet;

# COMMAND ----------

# MAGIC %sql
# MAGIC -- View a specific table with filtering
# MAGIC SELECT *
# MAGIC FROM main.silver_dev.connect_countryriskdet
# MAGIC LIMIT 10;

# COMMAND ----------

# MAGIC %md
# MAGIC # 7. Compare Row Counts Across Layers (Data Quality Check)

# COMMAND ----------

# Compare row counts from landing → conformance → silver for each source
sources = [
    ("connect_countryriskdet", f"{catalog}.{bronze_schema}.connect_countryriskdet_landing", 
     f"{catalog}.{bronze_schema}.connect_countryriskdet_conformance", 
     f"{catalog}.{silver_schema}.connect_countryriskdet"),
    ("pia_item", f"{catalog}.{bronze_schema}.pia_item_landing",
     f"{catalog}.{bronze_schema}.pia_item_conformance",
     f"{catalog}.{silver_schema}.pia_item"),
]

print("Row Count Comparison Across Layers:")
print("="*100)

for source_name, landing_table, conformance_table, silver_table in sources:
    try:
        landing_count = spark.sql(f"SELECT COUNT(*) as cnt FROM {landing_table}").collect()[0][0]
        conformance_count = spark.sql(f"SELECT COUNT(*) as cnt FROM {conformance_table}").collect()[0][0]
        silver_count = spark.sql(f"SELECT COUNT(*) as cnt FROM {silver_table}").collect()[0][0]
        
        print(f"\n{source_name}:")
        print(f"  Landing:     {landing_count:>10,} rows")
        print(f"  Conformance: {conformance_count:>10,} rows")
        print(f"  Silver:      {silver_count:>10,} rows")
    except Exception as e:
        print(f"\n{source_name}: Error - {e}")

print("\n" + "="*100)

# COMMAND ----------

# MAGIC %md
# MAGIC # 8. External Location vs Framework Output Comparison

# COMMAND ----------

comparison_rows = []

for source_name, landing_table, conformance_table, silver_table in sources:
    row: dict[str, Any] = {
        "source_name": source_name,
        "landing_table": landing_table,
        "conformance_table": conformance_table,
        "silver_table": silver_table,
        "landing_count": None,
        "conformance_count": None,
        "silver_count": None,
        "silver_location": None,
        "under_external_location": None,
        "status": "UNKNOWN",
    }

    try:
        row["landing_count"] = spark.sql(f"SELECT COUNT(*) AS c FROM {landing_table}").collect()[0]["c"]
        row["conformance_count"] = spark.sql(f"SELECT COUNT(*) AS c FROM {conformance_table}").collect()[0]["c"]
        row["silver_count"] = spark.sql(f"SELECT COUNT(*) AS c FROM {silver_table}").collect()[0]["c"]

        detail = spark.sql(f"DESCRIBE DETAIL {silver_table}").collect()[0].asDict()
        row["silver_location"] = str(detail.get("location", ""))
        row["under_external_location"] = str(row["silver_location"]).startswith(external_location_path)

        counts_monotonic = (
            row["landing_count"] is not None
            and row["conformance_count"] is not None
            and row["silver_count"] is not None
        )

        if counts_monotonic and row["under_external_location"]:
            row["status"] = "MATCHED"
        elif counts_monotonic:
            row["status"] = "ROW_COUNTS_OK_LOCATION_DIFFERS"
        else:
            row["status"] = "INCOMPLETE"
    except Exception as exc:
        row["status"] = f"ERROR: {exc}"

    comparison_rows.append(row)

comparison_df = spark.createDataFrame(comparison_rows)
print("Comparison summary (expected framework output vs actual external-location placement):")
comparison_df.display()
